# CAS Exam 5: Case Outstanding Techniques

This notebook demonstrates both case outstanding techniques from your formula sheet, while leveraging `chainladder` functionality as much as possible.

Data used:
- `chainladder/utils/data/friedland_us_industry_auto_case.csv`

Methods shown:
1. Case Outstanding Technique #1 (develop case outstanding + paid-on-case style projection).
2. Case Outstanding Technique #2 (case outstanding projection using paid/reported CDF relationship).


## Formula-Sheet Summary

Technique #1 (case development + incremental paid):
- Remaining-in-case ratio = Current Case Outstanding / Prior Case Outstanding
- Paid-on-case ratio = Incremental Paid Claims / Prior Case Outstanding
- Project future case outstanding and incremental paid iteratively by age.

Technique #2 (industry CDF combination):
- Combine paid and reported CDFs into a case outstanding development factor.
- One common form is:
  Case OS Dev Factor = 1 + ((Reported CDF - 1) x Paid CDF) / (Paid CDF - Reported CDF)
- Multiply latest case outstanding by this factor to project unpaid claims.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import chainladder as cl
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import (
    build_triangle,
    fit_development,
    run_case_outstanding_chainladder,
    run_case_outstanding_friedland,
    triangle_to_frame,
)

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto_case.csv'
raw = pd.read_csv(DATA_PATH).sort_values(['Accident Year', 'Calendar Year']).copy()

raw['Paid Cumulative'] = raw.groupby('Accident Year')['Incremental Paid Claims'].cumsum()
raw['Reported Cumulative'] = raw['Paid Cumulative'] + raw['Case Outstanding']

case_os_triangle = build_triangle(
    raw,
    origin_col='Accident Year',
    development_col='Calendar Year',
    value_cols=['Case Outstanding'],
    cumulative=False,
)
paid_incremental_triangle = build_triangle(
    raw,
    origin_col='Accident Year',
    development_col='Calendar Year',
    value_cols=['Incremental Paid Claims'],
    cumulative=False,
)
paid_cumulative_triangle = build_triangle(
    raw,
    origin_col='Accident Year',
    development_col='Calendar Year',
    value_cols=['Paid Cumulative'],
    cumulative=True,
)
reported_cumulative_triangle = build_triangle(
    raw,
    origin_col='Accident Year',
    development_col='Calendar Year',
    value_cols=['Reported Cumulative'],
    cumulative=True,
)

{'rows': len(raw), 'case_triangle_shape': case_os_triangle.shape, 'paid_cum_shape': paid_cumulative_triangle.shape}


In [ ]:
latest_case = case_os_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_paid = paid_cumulative_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_reported = reported_cumulative_triangle.latest_diagonal.to_frame().iloc[:, 0]

case_snapshot = pd.DataFrame(
    {
        'LatestCaseOutstanding': latest_case.values,
        'LatestPaidCumulative': latest_paid.values,
        'LatestReportedCumulative': latest_reported.values,
    },
    index=latest_case.index.year,
)
case_snapshot.index.name = 'AccidentYear'
case_snapshot


## Technique #1: Develop Case Outstanding and Project Incremental Paid

This section uses two implementations:
- A Friedland-style explicit ratio implementation (`run_case_outstanding_friedland`).
- A chainladder-native implementation using `chainladder.CaseOutstanding` (`run_case_outstanding_chainladder`).

Both use case and paid triangles, but the chainladder-native path keeps more of the projection workflow inside `chainladder` objects.


In [ ]:
friedland_result, friedland_summary, friedland_artifacts = run_case_outstanding_friedland(
    case_os_triangle,
    paid_incremental_triangle,
)

co_chainladder_result, co_chainladder_summary, co_chainladder_artifacts = run_case_outstanding_chainladder(
    paid_cumulative_triangle,
    reported_cumulative_triangle,
)

ratio_table = friedland_artifacts['ratio_table'].copy()
ratio_table.head(12), friedland_summary.head(8), co_chainladder_summary.head(8)


## Technique #1 Comparison: Explicit Ratios vs Chainladder CaseOutstanding

The comparison below highlights total-level differences across implementations.
Differences are expected because the two pipelines do not estimate exactly the same internal quantity in the same way.


In [ ]:
tech1_totals = pd.DataFrame(
    [
        {
            'Method': friedland_result.method_name,
            'LatestTotal': friedland_result.latest_reported_total,
            'UltimateTotal': friedland_result.ultimate_total,
            'IBNRTotal': friedland_result.ibnr_total,
        },
        {
            'Method': co_chainladder_result.method_name,
            'LatestTotal': co_chainladder_result.latest_reported_total,
            'UltimateTotal': co_chainladder_result.ultimate_total,
            'IBNRTotal': co_chainladder_result.ibnr_total,
        },
    ]
)

tech1_chainladder_patterns = pd.DataFrame({
    'Combined_LDF': pd.Series(co_chainladder_artifacts['selected_ldfs_by_age']['combined'].get('(All)', {})),
    'Paid_to_PriorCase_LDF': pd.Series(co_chainladder_artifacts['selected_ldfs_by_age']['paid_to_prior_case'].get('(All)', {})),
    'Case_to_PriorCase_LDF': pd.Series(co_chainladder_artifacts['selected_ldfs_by_age']['case_to_prior_case'].get('(All)', {})),
})

tech1_totals, tech1_chainladder_patterns


## Technique #2: Case Outstanding with Paid/Reported CDF Combination

This method uses industry-style paid and reported CDFs to build a case outstanding development factor by maturity.

Implementation steps:
1. Fit development to paid cumulative and reported cumulative triangles.
2. Get selected paid and reported CDFs by maturity.
3. Compute Case OS Dev Factor from the CDF relationship.
4. Apply factor to latest case outstanding to project unpaid and ultimate paid.


In [ ]:
paid_dev, _ = fit_development(paid_cumulative_triangle, average='volume', n_periods=-1)
reported_dev, _ = fit_development(reported_cumulative_triangle, average='volume', n_periods=-1)

paid_cdf_by_age = cl.Chainladder().fit(paid_dev).cdf_.to_frame().iloc[0]
reported_cdf_by_age = cl.Chainladder().fit(reported_dev).cdf_.to_frame().iloc[0]

case_long = triangle_to_frame(case_os_triangle, origin_as_datetime=False).reset_index()
case_col = [c for c in case_long.columns if c not in ['Total', 'origin', 'development', 'valuation']][0]
case_matrix = case_long.pivot(index='origin', columns='development', values=case_col).sort_index().sort_index(axis=1)
latest_age_by_ay = case_matrix.notna().iloc[:, ::-1].idxmax(axis=1).astype(int)
latest_age_by_ay.index = latest_age_by_ay.index.map(lambda x: int(getattr(x, 'year', int(str(x)[:4]))))

tech2_rows = []
for idx in latest_case.index:
    ay = int(idx.year)
    age = int(latest_age_by_ay.loc[ay])
    paid_cdf = float(paid_cdf_by_age.get(f'{age}-Ult', 1.0))
    reported_cdf = float(reported_cdf_by_age.get(f'{age}-Ult', 1.0))
    denom = paid_cdf - reported_cdf
    case_os_factor = 1.0 + ((reported_cdf - 1.0) * paid_cdf / denom) if abs(denom) > 1e-12 else 1.0

    latest_case_val = float(latest_case.loc[idx])
    latest_paid_val = float(latest_paid.loc[idx])
    latest_incurred_val = float(latest_reported.loc[idx])
    projected_unpaid = latest_case_val * case_os_factor
    projected_ultimate_paid = latest_paid_val + projected_unpaid
    additional_ibnr_above_case = projected_unpaid - latest_case_val

    tech2_rows.append(
        {
            'AccidentYear': ay,
            'LatestAge': age,
            'PaidCDF': paid_cdf,
            'ReportedCDF': reported_cdf,
            'CaseOSDevFactor': case_os_factor,
            'LatestCaseOutstanding': latest_case_val,
            'LatestPaidCumulative': latest_paid_val,
            'LatestReportedCumulative': latest_incurred_val,
            'ProjectedUnpaid': projected_unpaid,
            'ProjectedUltimatePaid': projected_ultimate_paid,
            'AdditionalIBNRAboveCase': additional_ibnr_above_case,
        }
    )

tech2_projection = pd.DataFrame(tech2_rows).set_index('AccidentYear').sort_index()
tech2_projection.loc['Total'] = tech2_projection.sum()
tech2_projection


## Technique #2 Diagnostic Tables

The tables below show the paid/reported CDF curves used in the combination formula, and the resulting case outstanding development factor by age.


In [ ]:
cdf_compare = pd.DataFrame(
    {
        'PaidCDF': paid_cdf_by_age,
        'ReportedCDF': reported_cdf_by_age,
    }
)
cdf_compare['CaseOSDevFactor'] = 1.0 + ((cdf_compare['ReportedCDF'] - 1.0) * cdf_compare['PaidCDF']) / (cdf_compare['PaidCDF'] - cdf_compare['ReportedCDF'])
cdf_compare


## Final Comparison and Assumptions

Key assumptions for case outstanding methods:
1. Case adequacy and claims processing remain reasonably stable.
2. Paid/reporting development patterns are representative for future emergence.
3. Mix, limits, and retention structure are sufficiently stable (or segmented).

Technique tradeoff summary:
- Technique #1 is process-intuitive and can be very useful when case and paid interaction is stable.
- Technique #2 is more benchmark-driven via paid/reported CDFs and can be robust when explicit case ratio calibration is noisy.


In [ ]:
final_comparison = pd.DataFrame(
    [
        {
            'Method': friedland_result.method_name,
            'ProjectedUltimate': friedland_result.ultimate_total,
            'ProjectedIBNR': friedland_result.ibnr_total,
            'ReferenceLatest': friedland_result.latest_reported_total,
        },
        {
            'Method': co_chainladder_result.method_name,
            'ProjectedUltimate': co_chainladder_result.ultimate_total,
            'ProjectedIBNR': co_chainladder_result.ibnr_total,
            'ReferenceLatest': co_chainladder_result.latest_reported_total,
        },
        {
            'Method': 'Case Outstanding Technique #2 (CDF Combination)',
            'ProjectedUltimate': float(tech2_projection.loc['Total', 'ProjectedUltimatePaid']),
            'ProjectedIBNR': float(tech2_projection.loc['Total', 'ProjectedUnpaid']),
            'ReferenceLatest': float(tech2_projection.loc['Total', 'LatestPaidCumulative']),
        },
    ]
)

impact_table = pd.DataFrame(
    [
        ['Increase in exposure', 'No material effect if average accident date remains stable'],
        ['Average accident date shifts forward', 'Can understate ultimates (more leverage at immature ages)'],
        ['Speedup in settlement rate', 'Can distort paid-on-case and paid-vs-reported relationships'],
        ['Increase in case adequacy', 'Can raise projected unpaid if not reflected in selected factors'],
        ['Change in product mix', 'Requires segmentation or re-selection of case/pattern factors'],
    ],
    columns=['Description', 'Impact on Case Outstanding Methods'],
)

final_comparison, impact_table
